# Test Meta-Model - Charité Dataset

This notebook tests a **stacking / meta-model** strategy using the probability outputs generated by the sensor-classifier models.

## Workflow
1. Load `sensor_classifier_decisions.csv` from the training notebook outputs.
2. Build a wide meta-feature matrix using per-sensor / per-classifier probabilities and hard predictions.
3. Train a meta-model with LOSO evaluation at the fold level.
4. Compare aggregate performance and save the results.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix, precision_recall_curve
)
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

In [ ]:
DATASET_NAME = 'Charité'
RESULTS_DIR = Path('../../outputs/charite_results')
DECISIONS_PATH = RESULTS_DIR / 'sensor_classifier_decisions.csv'
BEST_SENSOR_PATH = RESULTS_DIR / 'sensor_best_model_results.csv'
META_RESULTS_PATH = RESULTS_DIR / 'metamodel_loso_results.csv'
META_SUMMARY_PATH = RESULTS_DIR / 'metamodel_summary.csv'

required_paths = [DECISIONS_PATH, BEST_SENSOR_PATH]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Missing required files. Run the fusion section of 04_loso_pipeline_and_training.ipynb first:\n' + '\n'.join(missing)
    )

decision_df = pd.read_csv(DECISIONS_PATH)
sensor_best_df = pd.read_csv(BEST_SENSOR_PATH)

print(f'Dataset: {DATASET_NAME}')
print(f'Decision rows: {len(decision_df):,}')
print(f'Sensor-best rows: {len(sensor_best_df):,}')
print('Sensors:', sorted(decision_df['sensor'].unique()))
print('Classifiers:', sorted(decision_df['classifier'].unique()))

In [ ]:
def sanitize_name(text):
    return re.sub(r'[^0-9a-zA-Z_]+', '_', text).strip('_').lower()

decision_df['sensor_classifier'] = decision_df['sensor'] + ' | ' + decision_df['classifier']
meta_index = ['fold', 'window_index', 'y_true']

proba_wide = decision_df.pivot_table(
    index=meta_index, columns='sensor_classifier', values='y_proba', aggfunc='first'
)
pred_wide = decision_df.pivot_table(
    index=meta_index, columns='sensor_classifier', values='y_pred', aggfunc='first'
)

proba_wide.columns = [f"proba__{sanitize_name(col)}" for col in proba_wide.columns]
pred_wide.columns = [f"pred__{sanitize_name(col)}" for col in pred_wide.columns]

meta_df = pd.concat([proba_wide, pred_wide], axis=1).reset_index()
proba_cols = [col for col in meta_df.columns if col.startswith('proba__')]
pred_cols = [col for col in meta_df.columns if col.startswith('pred__')]

meta_df['proba_mean'] = meta_df[proba_cols].mean(axis=1)
meta_df['proba_max'] = meta_df[proba_cols].max(axis=1)
meta_df['proba_std'] = meta_df[proba_cols].std(axis=1).fillna(0.0)
feature_cols = proba_cols + pred_cols + ['proba_mean', 'proba_max', 'proba_std']

print(f'Meta-feature matrix: {meta_df.shape}')
print(f'Probability features: {len(proba_cols)}')
print(f'Hard-prediction features: {len(pred_cols)}')
print(f'Total meta-features used: {len(feature_cols)}')
meta_df.head()

In [ ]:
fold_summary = (meta_df.groupby('fold')
                .agg(n_windows=('window_index', 'count'),
                     fog_windows=('y_true', 'sum'))
                .assign(fog_rate=lambda df: df['fog_windows'] / df['n_windows']))
fold_summary

In [ ]:
def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_acc': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'cm': cm
    }

def tune_threshold(model, X_train, y_train, groups_train):
    unique_groups = pd.Series(groups_train).nunique()
    if unique_groups < 2:
        return 0.5

    cv_inner = GroupKFold(n_splits=min(5, unique_groups))
    y_proba_oof = cross_val_predict(
        clone(model), X_train, y_train, groups=groups_train,
        cv=cv_inner, method='predict_proba', n_jobs=1
    )[:, 1]

    precision, recall, thresholds = precision_recall_curve(y_train, y_proba_oof)
    f1_values = 2 * precision * recall / (precision + recall + 1e-9)
    if len(thresholds) == 0:
        return 0.5
    return float(np.clip(thresholds[np.argmax(f1_values[:-1])], 0.05, 0.95))

meta_models = {
    'Meta Logistic Regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ]),
    'Meta Random Forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('classifier', RandomForestClassifier(
            n_estimators=300,
            max_depth=5,
            min_samples_leaf=10,
            class_weight='balanced',
            random_state=42,
            n_jobs=1
        ))
    ])
}

print('Meta-models to test:')
for name in meta_models:
    print(f'  - {name}')

In [ ]:
results = []
fold_names = sorted(meta_df['fold'].unique())

for test_fold in fold_names:
    train_mask = meta_df['fold'] != test_fold
    test_mask = meta_df['fold'] == test_fold

    X_train = meta_df.loc[train_mask, feature_cols]
    y_train = meta_df.loc[train_mask, 'y_true']
    groups_train = meta_df.loc[train_mask, 'fold']

    X_test = meta_df.loc[test_mask, feature_cols]
    y_test = meta_df.loc[test_mask, 'y_true']

    for model_name, model in meta_models.items():
        threshold = tune_threshold(model, X_train, y_train, groups_train)
        fitted_model = clone(model)
        fitted_model.fit(X_train, y_train)
        y_proba = fitted_model.predict_proba(X_test)[:, 1]
        y_pred = (y_proba >= threshold).astype(int)

        metrics = compute_metrics(y_test, y_pred)
        results.append({
            'model': model_name,
            'fold': test_fold,
            'threshold': threshold,
            'accuracy': metrics['accuracy'],
            'balanced_acc': metrics['balanced_acc'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1': metrics['f1'],
            'cm': metrics['cm'].tolist(),
            'n_test': int(len(y_test)),
            'n_fog_test': int(y_test.sum())
        })

results_df = pd.DataFrame(results)
results_df.head()

In [ ]:
summary_rows = []

for model_name, group_df in results_df.groupby('model'):
    cm_total = np.zeros((2, 2), dtype=int)
    for cm in group_df['cm']:
        cm_total += np.array(cm, dtype=int)

    tn, fp, fn, tp = cm_total.ravel()
    total = cm_total.sum()
    acc = (tp + tn) / total if total else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    tnr = tn / (tn + fp) if (tn + fp) else 0.0
    bal_acc = (rec + tnr) / 2

    summary_rows.append({
        'Model': model_name,
        'Accuracy': acc,
        'Balanced Acc': bal_acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'Mean Threshold': group_df['threshold'].mean(),
        'Total Samples': int(total)
    })

summary_df = pd.DataFrame(summary_rows).sort_values('F1', ascending=False).reset_index(drop=True)
results_df.to_csv(META_RESULTS_PATH, index=False)
summary_df.to_csv(META_SUMMARY_PATH, index=False)

print('Saved:')
print(f'  - {META_RESULTS_PATH}')
print(f'  - {META_SUMMARY_PATH}')
summary_df